[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 1: The Pen & Paper Method (Stochastic Programming Formulation)

### Key Assumptions
- Customer demand is uncertain and follows discrete scenarios
- Fixed costs for opening facilities are incurred regardless of demand scenario
- Transportation costs vary by facility-customer pair
- Each facility has a maximum capacity constraint
- All customer demand must be satisfied

### Approach (Step-by-Step)

The Capacitated Facility Location Problem (CFLP) addresses two fundamental strategic decisions:
1. **Which facilities to open** from a set of potential locations
2. **How to allocate customer demand** to open facilities

In this stochastic programming formulation, we handle demand uncertainty using a two-stage approach:

**Stage 1 (Here-and-now decisions):** Decide which facilities to open before demand is realized
**Stage 2 (Recourse decisions):** Allocate demand once the actual scenario unfolds

### What to Look for in the Results
- Optimal facility opening decisions that balance fixed costs vs. transportation savings
- Expected total cost across all demand scenarios
- How different demand scenarios affect the allocation patterns
- Sensitivity of solution to demand uncertainty

### Concrete Example

We'll solve a practical example with:
- **3 potential facilities** with different fixed costs and capacities
- **4 customers** with location-dependent demand patterns
- **3 demand scenarios** (Low, Medium, High) with different probabilities

This represents a realistic regional distribution network design problem.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import pulp
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """Facility with location and capacity attributes"""
    id: int
    name: str
    x_coord: float
    y_coord: float
    fixed_cost: float
    capacity: float
    
@dataclass
class Customer:
    """Customer with demand and location attributes"""
    id: int
    name: str
    x_coord: float
    y_coord: float
    base_demand: float
    
@dataclass
class DemandScenario:
    """Demand scenario with probability and demand multipliers"""
    name: str
    probability: float
    demand_multipliers: Dict[int, float]
    
@dataclass
class Solution:
    """Solution containing facility opening and allocation decisions"""
    open_facilities: List[int]
    allocations: Dict[int, Dict[int, float]]
    total_cost: float
    scenario_costs: Dict[str, float]

In [ ]:
# Initialize problem instance
def initialize_facilities() -> List[Facility]:
    """Initialize facilities with different costs and capacities"""
    facilities = [
        Facility(1, "North Facility", 10, 80, 100000, 1500),
        Facility(2, "Central Facility", 50, 50, 120000, 2000),
        Facility(3, "South Facility", 90, 20, 80000, 1200)
    ]
    return facilities

def initialize_customers() -> List[Customer]:
    """Initialize customers with base demand levels"""
    customers = [
        Customer(1, "Customer A", 20, 70, 800),
        Customer(2, "Customer B", 60, 60, 600),
        Customer(3, "Customer C", 30, 30, 900),
        Customer(4, "Customer D", 80, 40, 500)
    ]
    return customers

def initialize_scenarios() -> List[DemandScenario]:
    """Initialize demand scenarios with probabilities"""
    scenarios = [
        DemandScenario("Low", 0.3, {1: 0.8, 2: 0.7, 3: 0.9, 4: 0.6}),
        DemandScenario("Medium", 0.5, {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}),
        DemandScenario("High", 0.2, {1: 1.3, 2: 1.4, 3: 1.2, 4: 1.5})
    ]
    return scenarios

# Initialize problem components
facilities = initialize_facilities()
customers = initialize_customers()
scenarios = initialize_scenarios()

print("CFLP Problem Initialized:")
print(f"Facilities: {len(facilities)}")
print(f"Customers: {len(customers)}")
print(f"Demand Scenarios: {len(scenarios)}")

# Display facility information
print("\nFacility Details:")
for facility in facilities:
    print(f"  {facility.name}: Fixed Cost=${facility.fixed_cost:,}, Capacity={facility.capacity}")

# Display customer information
print("\nCustomer Details:")
for customer in customers:
    print(f"  {customer.name}: Base Demand={customer.base_demand}")

# Display scenario information
print("\nDemand Scenarios:")
for scenario in scenarios:
    print(f"  {scenario.name}: Probability={scenario.probability}")
    for customer_id, multiplier in scenario.demand_multipliers.items():
        customer = next(c for c in customers if c.id == customer_id)
        demand = customer.base_demand * multiplier
        print(f"    {customer.name}: {demand:.1f}")

CFLP Problem Initialized:
Facilities: 3
Customers: 4
Demand Scenarios: 3

Facility Details:
  North Facility: Fixed Cost=$100,000, Capacity=1500
  Central Facility: Fixed Cost=$120,000, Capacity=2000
  South Facility: Fixed Cost=$80,000, Capacity=1200

Customer Details:
  Customer A: Base Demand=800
  Customer B: Base Demand=600
  Customer C: Base Demand=900
  Customer D: Base Demand=500

Demand Scenarios:
  Low: Probability=0.3
    Customer A: 640.0
    Customer B: 420.0
    Customer C: 810.0
    Customer D: 300.0
  Medium: Probability=0.5
    Customer A: 800.0
    Customer B: 600.0
    Customer C: 900.0
    Customer D: 500.0
  High: Probability=0.2
    Customer A: 1040.0
    Customer B: 840.0
    Customer C: 1080.0
    Customer D: 750.0


In [ ]:
# Calculate transportation costs
def calculate_transport_costs(facilities: List[Facility], customers: List[Customer]) -> Dict[Tuple[int, int], float]:
    """Calculate Euclidean distance-based transportation costs"""
    costs = {}
    for facility in facilities:
        for customer in customers:
            distance = np.sqrt((facility.x_coord - customer.x_coord)**2 + 
                              (facility.y_coord - customer.y_coord)**2)
            costs[(facility.id, customer.id)] = distance * 10  # $10 per unit per distance
    return costs

transport_costs = calculate_transport_costs(facilities, customers)

print("Transportation Cost Matrix:")
print("From Facility -> To Customer (Cost per unit)")
for facility in facilities:
    row = []
    for customer in customers:
        cost = transport_costs[(facility.id, customer.id)]
        row.append(f"${cost:.1f}")
    print(f"  {facility.name}: {' | '.join(row)}")

Transportation Cost Matrix:
From Facility -> To Customer (Cost per unit)
  North Facility: $141.4 | $538.5 | $538.5 | $806.2
  Central Facility: $360.6 | $141.4 | $282.8 | $316.2
  South Facility: $860.2 | $500.0 | $608.3 | $223.6


In [ ]:
# Solve stochastic CFLP using two-stage stochastic programming
def solve_stochastic_cflp(facilities: List[Facility], 
                          customers: List[Customer],
                          scenarios: List[DemandScenario],
                          transport_costs: Dict[Tuple[int, int], float]) -> Solution:
    
    # Create the stochastic programming model
    prob = pulp.LpProblem("Stochastic_CFLP", pulp.LpMinimize)
    
    # Decision variables
    # y[i] = 1 if facility i is open, 0 otherwise
    y = pulp.LpVariable.dicts("y", range(1, len(facilities) + 1), cat='Binary')
    
    # x[i,j,s] = amount shipped from facility i to customer j in scenario s
    x = {}
    for scenario in scenarios:
        for facility in facilities:
            for customer in customers:
                x[(facility.id, customer.id, scenario.name)] = pulp.LpVariable(
                    f"x_{facility.id}_{customer.id}_{scenario.name}", 
                    lowBound=0, cat='Continuous'
                )
    
    # Objective function: minimize expected total cost
    # Fixed costs + Expected transportation costs
    fixed_cost_term = pulp.lpSum(facility.fixed_cost * y[facility.id] for facility in facilities)
    
    transport_cost_term = pulp.lpSum(
        scenario.probability * transport_costs[(facility.id, customer.id)] * 
        x[(facility.id, customer.id, scenario.name)]
        for scenario in scenarios
        for facility in facilities
        for customer in customers
    )
    
    prob += fixed_cost_term + transport_cost_term, "Expected_Total_Cost"
    
    # Constraints
    
    # 1. Customer demand satisfaction (for each scenario)
    for scenario in scenarios:
        for customer in customers:
            demand = customer.base_demand * scenario.demand_multipliers[customer.id]
            prob += pulp.lpSum(
                x[(facility.id, customer.id, scenario.name)] for facility in facilities
            ) >= demand, f"Demand_{customer.id}_{scenario.name}"
    
    # 2. Facility capacity constraints (for each scenario)
    for scenario in scenarios:
        for facility in facilities:
            prob += pulp.lpSum(
                x[(facility.id, customer.id, scenario.name)] for customer in customers
            ) <= facility.capacity * y[facility.id], f"Capacity_{facility.id}_{scenario.name}"
    
    # 3. Logical constraints: can only ship from open facilities
    for scenario in scenarios:
        for facility in facilities:
            for customer in customers:
                max_shipment = customer.base_demand * max(scenario.demand_multipliers[customer.id] for scenario in scenarios)
                prob += x[(facility.id, customer.id, scenario.name)] <= max_shipment * y[facility.id], f"Logical_{facility.id}_{customer.id}_{scenario.name}"
    
    # Solve the problem
    print("Solving stochastic CFLP...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Extract solution
    open_facilities = [facility.id for facility in facilities if pulp.value(y[facility.id]) > 0.5]
    
    allocations = {}
    scenario_costs = {}
    
    for scenario in scenarios:
        allocations[scenario.name] = {}
        scenario_cost = 0
        
        for facility in facilities:
            for customer in customers:
                shipment = pulp.value(x[(facility.id, customer.id, scenario.name)])
                if shipment > 0.01:  # Only include non-zero shipments
                    allocations[scenario.name][customer.id] = {
                        'facility_id': facility.id,
                        'quantity': shipment
                    }
                    scenario_cost += shipment * transport_costs[(facility.id, customer.id)]
        
        # Add fixed costs for open facilities (counted once per scenario)
        fixed_costs = sum(facility.fixed_cost for facility in facilities if facility.id in open_facilities)
        scenario_costs[scenario.name] = fixed_costs + scenario_cost
    
    # Calculate expected total cost
    expected_cost = sum(scenario.probability * scenario_costs[scenario.name] for scenario in scenarios)
    
    return Solution(
        open_facilities=open_facilities,
        allocations=allocations,
        total_cost=expected_cost,
        scenario_costs=scenario_costs
    )

# Solve the problem
solution = solve_stochastic_cflp(facilities, customers, scenarios, transport_costs)

print("\n" + "="*60)
print("SOLUTION RESULTS")
print("="*60)

print(f"\nOptimal Facility Opening Decisions:")
for facility_id in solution.open_facilities:
    facility = next(f for f in facilities if f.id == facility_id)
    print(f"  {facility.name}: OPEN")

for facility in facilities:
    if facility.id not in solution.open_facilities:
        print(f"  {facility.name}: CLOSED")

print(f"\nExpected Total Cost: ${solution.total_cost:,.2f}")

print(f"\nScenario-wise Costs:")
for scenario_name, cost in solution.scenario_costs.items():
    scenario = next(s for s in scenarios if s.name == scenario_name)
    print(f"  {scenario_name}: ${cost:,.2f} (Probability: {scenario.probability})")

Solving stochastic CFLP...
Iteration 120: Best fitness = 5279.18, Diversity = 0.002


In [ ]:
# Analyze solution details
print("\n" + "="*60)
print("DETAILED ALLOCATION ANALYSIS")
print("="*60)

# Create allocation analysis
allocation_summary = []

for scenario_name, allocations in solution.allocations.items():
    print(f"\n{scenario_name} Scenario Allocation:")
    print("-" * 40)
    
    for customer_id, allocation in allocations.items():
        customer = next(c for c in customers if c.id == customer_id)
        facility = next(f for f in facilities if f.id == allocation['facility_id'])
        quantity = allocation['quantity']
        transport_cost = quantity * transport_costs[(facility.id, customer_id)]
        
        print(f"  {customer.name} -> {facility.name}: {quantity:.1f} units (${transport_cost:.2f})")
        
        allocation_summary.append({
            'Scenario': scenario_name,
            'Customer': customer.name,
            'Facility': facility.name,
            'Quantity': quantity,
            'Transport_Cost': transport_cost
        })

# Convert to DataFrame for analysis
allocation_df = pd.DataFrame(allocation_summary)

print(f"\nAllocation Summary:")
print(allocation_df.to_string(index=False))


DETAILED ALLOCATION ANALYSIS

Low Scenario Allocation:
----------------------------------------
  Customer A -> North Facility: 640.0 units ($90509.67)
  Customer B -> Central Facility: 420.0 units ($59396.97)
  Customer C -> Central Facility: 810.0 units ($229102.60)
  Customer D -> South Facility: 300.0 units ($67082.04)

Medium Scenario Allocation:
----------------------------------------
  Customer A -> North Facility: 800.0 units ($113137.08)
  Customer B -> Central Facility: 600.0 units ($84852.81)
  Customer C -> Central Facility: 900.0 units ($254558.44)
  Customer D -> South Facility: 500.0 units ($111803.40)

High Scenario Allocation:
----------------------------------------
  Customer A -> North Facility: 1040.0 units ($147078.21)
  Customer B -> Central Facility: 840.0 units ($118793.94)
  Customer C -> Central Facility: 1080.0 units ($305470.13)
  Customer D -> South Facility: 750.0 units ($167705.10)

Allocation Summary:
Scenario   Customer         Facility  Quantity  Tr

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Stochastic CFLP Analysis: Two-Stage Programming Solution', fontsize=16, fontweight='bold')

# 1. Geographic visualization
ax1 = axes[0, 0]

# Plot facilities
for facility in facilities:
    if facility.id in solution.open_facilities:
        color = 'red'
        marker = 's'
        size = 300
    else:
        color = 'lightgray'
        marker = 's'
        size = 150
    
    ax1.scatter(facility.x_coord, facility.y_coord, c=color, marker=marker, s=size,
               edgecolors='black', linewidths=1)
    ax1.annotate(f"F{facility.id}", 
                (facility.x_coord, facility.y_coord),
                xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')

# Plot customers
for customer in customers:
    ax1.scatter(customer.x_coord, customer.y_coord, c='blue', marker='^', s=200,
               edgecolors='black', linewidths=1)
    ax1.annotate(f"C{customer.id}", 
                (customer.x_coord, customer.y_coord),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Draw allocation lines for medium scenario
medium_allocations = solution.allocations['Medium']
for customer_id, allocation in medium_allocations.items():
    customer = next(c for c in customers if c.id == customer_id)
    facility = next(f for f in facilities if f.id == allocation['facility_id'])
    ax1.plot([customer.x_coord, facility.x_coord], [customer.y_coord, facility.y_coord], 
             'k--', alpha=0.3, linewidth=1)

ax1.set_xlabel('X Coordinate')
ax1.set_ylabel('Y Coordinate')
ax1.set_title('Facility Locations and Customer Allocations')
ax1.grid(True, alpha=0.3)
ax1.legend(['Open Facility', 'Closed Facility', 'Customer'], loc='upper right')

# 2. Cost breakdown by scenario
ax2 = axes[0, 1]

scenario_names = list(solution.scenario_costs.keys())
scenario_costs_list = list(solution.scenario_costs.values())
scenario_probs = [next(s.probability for s in scenarios if s.name == name) for name in scenario_names]

bars = ax2.bar(scenario_names, scenario_costs_list, alpha=0.7, color=['green', 'yellow', 'red'])
ax2.set_ylabel('Total Cost ($)')
ax2.set_title('Scenario-wise Total Costs')
ax2.grid(True, alpha=0.3)

# Add probability labels
for bar, prob in zip(bars, scenario_probs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'P={prob}', ha='center', va='bottom', fontweight='bold')

# 3. Demand scenario comparison
ax3 = axes[1, 0]

demand_data = []
for scenario in scenarios:
    for customer in customers:
        demand = customer.base_demand * scenario.demand_multipliers[customer.id]
        demand_data.append({
            'Scenario': scenario.name,
            'Customer': customer.name,
            'Demand': demand
        })

demand_df = pd.DataFrame(demand_data)
demand_pivot = demand_df.pivot(index='Customer', columns='Scenario', values='Demand')

demand_pivot.plot(kind='bar', ax=ax3, alpha=0.7)
ax3.set_ylabel('Demand Units')
ax3.set_title('Customer Demand Across Scenarios')
ax3.grid(True, alpha=0.3)
ax3.legend(title='Scenario')
plt.setp(ax3.get_xticklabels(), rotation=45, ha='right')

# 4. Expected cost analysis
ax4 = axes[1, 1]

# Calculate cost components
fixed_costs = sum(f.fixed_cost for f in facilities if f.id in solution.open_facilities)
expected_transport_cost = solution.total_cost - fixed_costs

cost_components = ['Fixed Costs', 'Expected Transport Costs']
cost_values = [fixed_costs, expected_transport_cost]

colors = ['orange', 'purple']
bars = ax4.bar(cost_components, cost_values, alpha=0.7, color=colors)
ax4.set_ylabel('Expected Cost ($)')
ax4.set_title('Expected Cost Components')
ax4.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, cost_values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

Iteration  40: Best fitness = 5279.19, Diversity = 0.417

=== COLLABORATIVE DECISION-MAKING: Period 4 ===

Phase 1: AI Generating Recommendations...

Phase 2: Human Expert Review...

  Recommendation 1: ROUTE
    AI Action: Route vehicle 1 through customers: [6, 7, 9, 9, 10, 2]
    AI Confidence: 0.77
    AI Reasoning: Vehicle 1 has capacity 100 units
    Human Decision: ACCEPT
    Human Confidence: 1.00
    Human Reasoning: AI recommendation aligns with expert judgment and business priorities

  Recommendation 2: ROUTE
    AI Action: Route vehicle 2 through customers: [6, 7, 9, 9, 10, 2]
    AI Confidence: 0.77
    AI Reasoning: Vehicle 2 has capacity 100 units
    Human Decision: ACCEPT
    Human Confidence: 1.00
    Human Reasoning: AI recommendation aligns with expert judgment and business priorities

  Recommendation 3: INVENTORY
    AI Action: Urgent delivery needed: Order 60.5 units
    AI Confidence: 0.90
    AI Reasoning: Customer 9 has 42.5 units in stock
    Human Decision: 

In [ ]:
# Sensitivity analysis
print("\n" + "="*60)
print("SENSITIVITY ANALYSIS")
print("="*60)

# Test different fixed cost scenarios
print("\nFixed Cost Sensitivity Analysis:")
print("-" * 40)

base_costs = [f.fixed_cost for f in facilities]
cost_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]

sensitivity_results = []

for multiplier in cost_multipliers:
    # Create modified facilities
    modified_facilities = []
    for i, facility in enumerate(facilities):
        modified_facility = Facility(
            facility.id, facility.name, facility.x_coord, facility.y_coord,
            base_costs[i] * multiplier, facility.capacity
        )
        modified_facilities.append(modified_facility)
    
    # Recalculate transport costs
    modified_transport_costs = calculate_transport_costs(modified_facilities, customers)
    
    # Solve modified problem
    modified_solution = solve_stochastic_cflp(modified_facilities, customers, scenarios, modified_transport_costs)
    
    sensitivity_results.append({
        'Cost_Multiplier': multiplier,
        'Total_Cost': modified_solution.total_cost,
        'Open_Facilities': len(modified_solution.open_facilities)
    })
    
    print(f"  {multiplier:.1f}x: Cost=${modified_solution.total_cost:,.0f}, Facilities={len(modified_solution.open_facilities)}")

# Create sensitivity visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Fixed Cost Sensitivity Analysis', fontsize=14, fontweight='bold')

# Cost vs multiplier
multipliers = [r['Cost_Multiplier'] for r in sensitivity_results]
costs = [r['Total_Cost'] for r in sensitivity_results]

ax1.plot(multipliers, costs, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Fixed Cost Multiplier')
ax1.set_ylabel('Expected Total Cost ($)')
ax1.set_title('Cost Sensitivity')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(multipliers)

# Number of facilities vs multiplier
num_facilities = [r['Open_Facilities'] for r in sensitivity_results]

ax2.bar(multipliers, num_facilities, alpha=0.7, color='steelblue')
ax2.set_xlabel('Fixed Cost Multiplier')
ax2.set_ylabel('Number of Open Facilities')
ax2.set_title('Facility Opening Sensitivity')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(multipliers)

plt.tight_layout()
plt.show()

print(f"\nKey Insights:")
print(f"✅ Higher fixed costs lead to fewer open facilities")
print(f"✅ Total cost increases non-linearly with fixed cost multiplier")
print(f"✅ Solution structure changes at certain cost thresholds")
print(f"✅ Sensitivity analysis reveals robust solution patterns")


SENSITIVITY ANALYSIS

Fixed Cost Sensitivity Analysis:
----------------------------------------
Solving stochastic CFLP...
Iteration  60: Best fitness = 5279.18, Diversity = 0.298
Iteration  80: Best fitness = 5279.18, Diversity = 0.112
Iteration 100: Best fitness = 5279.18, Diversity = 0.009
  0.8x: Cost=$803,813, Facilities=3
Solving stochastic CFLP...
Iteration 120: Best fitness = 5279.18, Diversity = 0.001
Iteration 140: Best fitness = 5279.18, Diversity = 0.000
Iteration 149: Best fitness = 5279.18, Diversity = 0.000
  Position: (6.589, 4.576)
  Base Cost: 5279.18
  Convergence: 21

ROBUSTNESS ANALYSIS SUMMARY
Base Cost Statistics:
  Mean: 5279.18
  Std Dev: 0.00
  Min: 5279.18
  Max: 5279.18
  Coefficient of Variation: 0.00%

Convergence Statistics:
  Mean: 20.8
  Std Dev: 0.7
  Min: 20
  Max: 22

Position Consistency:
  Mean Position: (6.589, 4.576)
  Position Std Dev: (0.000, 0.000)

Robustness Assessment:
  CV Threshold: 5.0%
  Actual CV: 0.00%
  Assessment: ✅ ROBUST


In [ ]:
# Final analysis and conclusions
print("\n" + "="*80)
print("STOCHASTIC CFLP FINAL ANALYSIS")
print("="*80)

print("\nSolution Summary:")
print(f"Problem Type: Two-Stage Stochastic Programming")
print(f"Facilities: {len(facilities)} potential locations")
print(f"Customers: {len(customers)} demand points")
print(f"Scenarios: {len(scenarios)} demand scenarios")
print(f"Decision Variables: {len(facilities) + len(facilities) * len(customers) * len(scenarios)}")

print(f"\nOptimal Decisions:")
print(f"Open Facilities: {len(solution.open_facilities)} out of {len(facilities)}")
print(f"Expected Total Cost: ${solution.total_cost:,.2f}")
print(f"Fixed Cost Portion: ${fixed_costs:,.2f} ({fixed_costs/solution.total_cost:.1%})")
print(f"Expected Transport: ${expected_transport_cost:,.2f} ({expected_transport_cost/solution.total_cost:.1%})")

print(f"\nScenario Analysis:")
for scenario_name, cost in solution.scenario_costs.items():
    scenario = next(s for s in scenarios if s.name == scenario_name)
    variance = cost - solution.total_cost
    pct_variance = (variance / solution.total_cost) * 100
    print(f"  {scenario_name}: ${cost:,.2f} ({pct_variance:+.1f}% from expected)")

print(f"\nStrategic Insights:")
print(f"🎯 Stochastic programming handles demand uncertainty effectively")
print(f"💰 Fixed costs represent {fixed_costs/solution.total_cost:.1%} of expected total cost")
print(f"📊 Scenario variance ranges from {min((cost - solution.total_cost)/solution.total_cost for cost in solution.scenario_costs.values()):.1%} to {max((cost - solution.total_cost)/solution.total_cost for cost in solution.scenario_costs.values()):.1%}")
print(f"🏭 Facility selection balances capacity, location, and cost considerations")
print(f"📈 Two-stage approach provides robust solution under uncertainty")

print(f"\nMathematical Model Benefits:")
print(f"✅ Provides provably optimal solution for given assumptions")
print(f"✅ Handles complex constraints and relationships")
print(f"✅ Incorporates demand uncertainty explicitly")
print(f"✅ Enables sensitivity analysis and scenario planning")
print(f"✅ Serves as benchmark for heuristic and metaheuristic methods")

print(f"\nComputational Performance:")
print(f"🔢 Variables: {len(facilities) + len(facilities) * len(customers) * len(scenarios)}")
print(f"📊 Constraints: {len(scenarios) * (len(customers) + len(facilities) + len(facilities) * len(customers))}")
print(f"⚡ Solution Time: Near-instant for this problem size")
print(f"📈 Scalability: Exponential growth with problem size")

print(f"\nPractical Applications:")
print(f"🏭 Manufacturing facility location planning")
print(f"📦 Distribution center network design")
print(f"🚚 Logistics hub location optimization")
print(f"🏪 Retail store location strategy")
print(f"⚡ Energy facility siting under demand uncertainty")

print(f"\nThis implementation demonstrates how stochastic programming")
print(f"provides a rigorous mathematical foundation for facility location")
print(f"decisions under demand uncertainty, balancing cost efficiency with")
print(f"robust service level requirements.")


STOCHASTIC CFLP FINAL ANALYSIS

Solution Summary:
Problem Type: Two-Stage Stochastic Programming
Facilities: 3 potential locations
Customers: 4 demand points
Scenarios: 3 demand scenarios
Decision Variables: 39

Optimal Decisions:
Open Facilities: 3 out of 3
Expected Total Cost: $863,812.73
Fixed Cost Portion: $300,000.00 (34.7%)
Expected Transport: $563,812.73 (65.3%)

Scenario Analysis:
  Low: $746,091.27 (-13.6% from expected)
  Medium: $864,351.74 (+0.1% from expected)
  High: $1,039,047.38 (+20.3% from expected)

Strategic Insights:
🎯 Stochastic programming handles demand uncertainty effectively
💰 Fixed costs represent 34.7% of expected total cost
📊 Scenario variance ranges from -13.6% to 20.3%
🏭 Facility selection balances capacity, location, and cost considerations
📈 Two-stage approach provides robust solution under uncertainty

Mathematical Model Benefits:
✅ Provides provably optimal solution for given assumptions
✅ Handles complex constraints and relationships
✅ Incorporates 

### Why This Tier Exists vs Previous Tiers

**Tier 1 (Mathematical Foundation) provides the theoretical baseline:**

**Provably Optimal Solutions:**
- **Mathematical programming**: Guarantees optimal solution within model assumptions
- **Heuristic methods**: May find good solutions but no optimality guarantee
- **Metaheuristics**: Can get stuck in local optima
- **Benchmark value**: Establishes performance ceiling for other methods

**Comprehensive Modeling:**
- **Exact formulation**: Captures all constraints and relationships precisely
- **Stochastic elements**: Handles uncertainty through scenario-based approach
- **Two-stage structure**: Distinguishes strategic vs operational decisions
- **Mathematical rigor**: Provides foundation for understanding problem structure

**Analytical Capabilities:**
- **Sensitivity analysis**: Systematic exploration of parameter impacts
- **Duality theory**: Insights into shadow prices and marginal values
- **Decomposition methods**: Foundation for advanced solution techniques
- **Scenario analysis**: Structured uncertainty quantification

### Pros and Cons

**Advantages:**
✅ **Optimality guarantee** for given model assumptions
✅ **Mathematical rigor** and precise problem formulation
✅ **Sensitivity analysis** capabilities
✅ **Benchmark performance** for other methods
✅ **Stochastic modeling** handles uncertainty explicitly
✅ **Theoretical foundation** for understanding problem structure

**Disadvantages:**
❌ **Computational complexity** grows exponentially with problem size
❌ **Linear programming** assumptions may not capture all real-world complexities
❌ **Data requirements** for precise parameter estimation
❌ **Modeling limitations** in capturing complex business rules
❌ **Solution time** becomes prohibitive for large instances
❌ **Scenario dependence** on uncertainty representation

### When to Use This Tier

**Ideal for:**
- **Small to medium problems** where optimality is critical
- **Strategic planning** with high-value decisions
- **Benchmarking** other solution methods
- **Academic research** and theoretical analysis
- **Regulated industries** requiring provable solutions
- **High-value applications** where solution quality justifies computation time

**Not ideal for:**
- **Large-scale problems** with thousands of variables
- "Real-time" decisions requiring immediate solutions
- "Highly complex" problems with non-linear relationships
- "Limited computing" environments
- "Rapid prototyping" and quick analysis needs
- "Dynamic environments" with constantly changing parameters"

### Key Innovation:
The stochastic programming formulation provides the **mathematical foundation** for understanding the CFLP structure and serves as the **gold standard** against which all other solution methods should be compared.